# Notebook 9 — Experiments 16 and 17: Cross-Dataset and Explainability
**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

| Exp | What | Fills |
|---|---|---|
| 16 | Cross-dataset robustness, TweetEval to SemEval | Table 6 |
| 17 | SHAP, LIME and ethical risk ranking | Table 6 |

Experiment 16 retrains the final configuration on TweetEval and evaluates it on
SemEval without retuning. Not retuning is deliberate: an organisation deploying a
model to a new data source does not retrain it first, so retuning would measure
something other than deployment robustness.

Vocabulary overlap between TweetEval and SemEval is only 40.4% from the EDA, so a
performance drop is expected. The question this answers is whether the drop falls
evenly across linguistic subgroups or concentrates in the informal ones.

## Cell 1: Setup

In [1]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, scipy.sparse as sp, pickle, time, json
from sklearn.model_selection import GridSearchCV

!pip install -q shap lime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 31.0 MB/s eta 0:00:00
Mounted at /content/drive
thesis_utils loaded.
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 21.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## Cell 2: Load Configuration and Data

In [2]:
D = PATHS["data"]

with open(D / "final_model_config.json") as f:
    cfg = json.load(f)

BEST_MODEL       = cfg["model"]
BEST_FEATURE_SET = cfg["feature_set"]

print("Final model from Exp 10:")
print(f"  model       : {BEST_MODEL}")
print(f"  feature set : {BEST_FEATURE_SET}")
print(f"  macro F1    : {cfg['macro_f1']:.4f}")

tw_train = pd.read_parquet(D / "tw_train.parquet")
tw_test  = pd.read_parquet(D / "tw_test.parquet")
se_test  = pd.read_parquet(D / "se_test.parquet")

print(f"\nTweetEval train : {len(tw_train):,}")
print(f"TweetEval test  : {len(tw_test):,}")
print(f"SemEval test    : {len(se_test):,}")

Final model from Exp 10:
  model       : LogisticRegression
  feature set : Hybrid feature set
  macro F1    : 0.5878

TweetEval train : 45,615
TweetEval test  : 12,284
SemEval test    : 1,758


## Cell 3: Experiment 16 — Cross-Dataset Robustness

Fit the vectoriser on TweetEval training text, then transform SemEval test text
with that same vectoriser. SemEval words absent from the TweetEval vocabulary are
dropped, which is exactly what happens in deployment.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

def make_model(name):
    if name == "LogisticRegression":
        return LogisticRegression(C=1, max_iter=2000, solver="liblinear", random_state=SEED)
    if name == "LinearSVM":
        return CalibratedClassifierCV(
            LinearSVC(C=1, max_iter=3000, random_state=SEED, dual="auto"),
            cv=3, method="sigmoid")
    if name == "MultinomialNB":   return MultinomialNB(alpha=0.5)
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=5,
                             random_state=SEED, n_jobs=-1,
                             eval_metric="mlogloss", tree_method="hist")

print("="*62)
print("EXP 16 — CROSS-DATASET ROBUSTNESS")
print("="*62)
print("  Train: TweetEval   Test: SemEval-2014")
print("  Hyperparameters fixed from source. No retuning on target.\n")

vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95,
                      max_features=50_000, sublinear_tf=True)
X_tw_train = vec.fit_transform(tw_train["text_clean"])
X_tw_test  = vec.transform(tw_test["text_clean"])
X_se_test  = vec.transform(se_test["text_clean"])      # TweetEval vocabulary

y_tw_train = tw_train["sentiment"].values
y_tw_test  = tw_test["sentiment"].values
y_se_test  = se_test["sentiment"].values

model = make_model(BEST_MODEL)
NEEDS_ENC = BEST_MODEL == "XGBoost"

if NEEDS_ENC:
    le = LabelEncoder().fit(y_tw_train)
    model.fit(X_tw_train, le.transform(y_tw_train))
    pred_tw = le.inverse_transform(model.predict(X_tw_test))
    pred_se = le.inverse_transform(model.predict(X_se_test))
else:
    model.fit(X_tw_train, y_tw_train)
    pred_tw = model.predict(X_tw_test)
    pred_se = model.predict(X_se_test)

proba_tw = model.predict_proba(X_tw_test)
proba_se = model.predict_proba(X_se_test)

res_tw = evaluate_model(y_tw_test, pred_tw, proba_tw, label="TweetEval (source)")
res_se = evaluate_model(y_se_test, pred_se, proba_se, label="SemEval (target)")

gen_gap = res_tw["Macro F1"] - res_se["Macro F1"]

print(f"  Source Macro F1 (TweetEval) : {res_tw['Macro F1']:.4f}")
print(f"  Target Macro F1 (SemEval)   : {res_se['Macro F1']:.4f}")
print(f"  Generalisation Gap          : {gen_gap:+.4f}")
print(f"\n  OOV rate on SemEval: "
      f"{(X_se_test.sum(axis=1) == 0).sum()} documents had zero known tokens")

save_predictions("exp16", "FinalModel", y_tw_test, pred_tw, proba_tw,
                 tw_test["subgroup_primary"].values, dataset="TweetEval")
save_predictions("exp16", "FinalModel", y_se_test, pred_se, proba_se,
                 se_test["subgroup_primary"].values, dataset="SemEval")

with open(D / "crossdataset_vectorizer.pkl", "wb") as f:
    pickle.dump(vec, f)
save_model(model, "exp16", "CrossDatasetModel")

EXP 16 — CROSS-DATASET ROBUSTNESS
  Train: TweetEval   Test: SemEval-2014
  Hyperparameters fixed from source. No retuning on target.

  Source Macro F1 (TweetEval) : 0.5453
  Target Macro F1 (SemEval)   : 0.5038
  Generalisation Gap          : +0.0415

  OOV rate on SemEval: 0 documents had zero known tokens
  saved predictions -> exp16_FinalModel_TweetEval.parquet  (12,284 rows)
  saved predictions -> exp16_FinalModel_SemEval.parquet  (1,758 rows)
  saved model -> exp16_CrossDatasetModel.pkl


## Cell 4: Cross-Dataset Subgroup Breakdown

The headline generalisation gap hides the more important question: does the drop
affect all linguistic subgroups equally?

SemEval is 95% formal text, so its informal subgroups are very small. Where a
subgroup has too few instances the comparison is reported but flagged.

In [4]:
pred_tw_df = load_predictions("exp16", "FinalModel", "TweetEval")
pred_se_df = load_predictions("exp16", "FinalModel", "SemEval")

rep_tw = subgroup_report(pred_tw_df)
rep_se = subgroup_report(pred_se_df)

comp = rep_tw[["Subgroup", "Number of Samples", "Macro F1"]].merge(
    rep_se[["Subgroup", "Number of Samples", "Macro F1"]],
    on="Subgroup", how="outer", suffixes=(" TweetEval", " SemEval"))
comp["Subgroup Gap"] = comp["Macro F1 TweetEval"] - comp["Macro F1 SemEval"]

print("SUBGROUP-LEVEL GENERALISATION")
print("="*80)
print(comp.to_string(index=False))

print("\nWhere a SemEval subgroup has fewer than 30 instances the comparison")
print("is not reliable. SemEval is 95% formal text by design, so the informal")
print("subgroups are expected to be sparse.")

save_result_table(comp, "Table6a_CrossDataset_Subgroup")

SUBGROUP-LEVEL GENERALISATION
   Subgroup  Number of Samples TweetEval  Macro F1 TweetEval  Number of Samples SemEval  Macro F1 SemEval  Subgroup Gap
emoji-heavy                          696            0.543689                        NaN               NaN           NaN
     formal                        10398            0.536674                     1532.0          0.507883      0.028791
      other                         1116            0.570690                      218.0          0.475991      0.094699
    sarcasm                           14            0.570238                        7.0          0.303030      0.267208
slang-heavy                           60            0.681172                        1.0          0.000000      0.681172

Where a SemEval subgroup has fewer than 30 instances the comparison
is not reliable. SemEval is 95% formal text by design, so the informal
subgroups are expected to be sparse.
  saved table -> Table6a_CrossDataset_Subgroup.csv


PosixPath('/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/results/Table6a_CrossDataset_Subgroup.csv')

## Cell 5: Experiment 17 Part A — SHAP Global Analysis

SHAP values computed separately for each linguistic subgroup partition. The
question is not which features matter overall but whether the features driving
predictions in formal posts are the same ones driving predictions in sarcasm
posts.

Explainer choice follows model type: LinearExplainer for linear models,
TreeExplainer for tree models. Sample sizes are capped for runtime.

In [11]:
import numpy as np
import scipy.sparse as sp
from sklearn.preprocessing import MinMaxScaler

with open(D / "final_transformers.pkl", "rb") as f:
    transformers = pickle.load(f)

final_model = load_model("exp10", "FinalModel")

print(f"Feature set from Exp 10 : {cfg['feature_set']}")
print(f"Transformers saved      : {type(transformers)}")
if isinstance(transformers, tuple):
    print(f"  count: {len(transformers)}")

SOCIAL_COLS = ["emoji_density", "slang_ratio", "punct_intensity"]

# ── rebuild exactly the feature space the model was trained on ────
if isinstance(transformers, tuple) and len(transformers) == 3:
    # hybrid: word vectoriser, char vectoriser, scaler
    vec_w, vec_c, scaler = transformers
    Xw = vec_w.transform(tw_test["text_clean"])
    Xc = vec_c.transform(tw_test["text_clean"])
    Xn = scaler.transform(tw_test[SOCIAL_COLS].values)
    X_test_final = sp.hstack([Xw, Xc, sp.csr_matrix(Xn)]).tocsr()
    feature_names = np.concatenate([
        vec_w.get_feature_names_out(),
        np.array([f"char_{f}" for f in vec_c.get_feature_names_out()]),
        np.array(SOCIAL_COLS),
    ])

elif isinstance(transformers, tuple) and len(transformers) == 2:
    # TF-IDF + engineered features
    vec_w, scaler = transformers
    Xw = vec_w.transform(tw_test["text_clean"])
    Xn = scaler.transform(tw_test[SOCIAL_COLS].values)
    X_test_final = sp.hstack([Xw, sp.csr_matrix(Xn)]).tocsr()
    feature_names = np.concatenate([
        vec_w.get_feature_names_out(), np.array(SOCIAL_COLS)])

else:
    vec_w = transformers
    X_test_final = vec_w.transform(tw_test["text_clean"])
    feature_names = np.array(vec_w.get_feature_names_out())

print(f"\nFeature matrix : {X_test_final.shape}")
print(f"Feature names  : {len(feature_names):,}")

# ── coefficients ──────────────────────────────────────────────────
inner = final_model
if hasattr(final_model, "calibrated_classifiers_"):
    inner = final_model.calibrated_classifiers_[0].estimator

coefs = np.asarray(inner.coef_) if hasattr(inner, "coef_") else np.asarray(inner.feature_log_prob_)
print(f"Coefficients   : {coefs.shape}")

assert coefs.shape[1] == X_test_final.shape[1], (
    f"MISMATCH: model has {coefs.shape[1]:,} features, "
    f"matrix has {X_test_final.shape[1]:,}")
assert len(feature_names) == X_test_final.shape[1], "feature name count mismatch"
print("Shapes aligned.\n")

SHAP_SAMPLE = 300
baseline = np.asarray(X_test_final.mean(axis=0)).ravel()

def shap_for_subgroup(subgroup, max_n=SHAP_SAMPLE):
    pos = np.where(tw_test["subgroup_primary"].values == subgroup)[0]
    if len(pos) == 0:
        return None
    pos = pos[:max_n]
    X_dense = np.asarray(X_test_final[pos].todense())
    centred = X_dense - baseline
    total = np.zeros(X_dense.shape[1])
    for c in range(coefs.shape[0]):
        total += np.abs(centred * coefs[c]).mean(axis=0)
    return total / coefs.shape[0]

shap_profiles = {}
for sg in ["formal", "emoji-heavy", "slang-heavy", "sarcasm"]:
    n = int((tw_test["subgroup_primary"] == sg).sum())
    print(f"Computing SHAP for {sg} (n={n:,}) ...")
    prof = shap_for_subgroup(sg)
    if prof is not None:
        shap_profiles[sg] = prof
        top = np.argsort(prof)[-10:][::-1]
        print(f"  top 10: {', '.join(feature_names[top])}\n")

print(f"Done. Profiles for {len(shap_profiles)} subgroups.")

Feature set from Exp 10 : Hybrid feature set
Transformers saved      : <class 'tuple'>
  count: 3

Feature matrix : (12284, 80003)
Feature names  : 80,003
Coefficients   : (3, 80003)
Shapes aligned.

Computing SHAP for formal (n=10,398) ...
  top 10: punct_intensity, to, not, trump, char_ed , char_ a , char_ i , it, char_in , is

Computing SHAP for emoji-heavy (n=696) ...
  top 10: happy, punct_intensity, love, char_ i , good, it, slang_ratio, to, we, great

Computing SHAP for slang-heavy (n=60) ...
  top 10: slang_ratio, punct_intensity, love, lol, bad, char_ lol, char_ i , wtf, fire, nice

Computing SHAP for sarcasm (n=14) ...
  top 10: not, at all, not at, idiots, punct_intensity, great, fantastic, char_ no, char_s! , for sure

Done. Profiles for 4 subgroups.


## Cell 6: Cross-Subgroup SHAP Comparison

Features ranking in the top ten for formal posts but absent from the sarcasm top
ten are the mechanism behind the fairness gap: the model relies on signals that
sarcastic language systematically disrupts.

In [12]:
if "formal" in shap_profiles:
    formal_top = set(feature_names[np.argsort(shap_profiles["formal"])[-20:]])

    rows = []
    for sg, prof in shap_profiles.items():
        top20 = set(feature_names[np.argsort(prof)[-20:]])
        rows.append({
            "Subgroup":              sg,
            "Top 5 Features":        ", ".join(feature_names[np.argsort(prof)[-5:][::-1]]),
            "Shared With Formal":    len(top20 & formal_top),
            "Unique To This Subgroup": ", ".join(list(top20 - formal_top)[:5]),
            "Overlap With Formal %":  round(len(top20 & formal_top) / 20 * 100, 1),
        })

    shap_comp = pd.DataFrame(rows)
    print("SHAP FEATURE OVERLAP WITH FORMAL REFERENCE GROUP")
    print("="*90)
    print(shap_comp.to_string(index=False))

    save_result_table(shap_comp, "Table6b_SHAP_Subgroup_Comparison")

    np.save(PATHS["results"] / "shap_profiles.npy", shap_profiles, allow_pickle=True)
    np.save(PATHS["results"] / "shap_feature_names.npy", feature_names)
    print("\nSHAP profiles saved for reuse in Notebook 15 (error taxonomy).")
else:
    print("SHAP unavailable for the formal reference group. Skipping comparison.")

SHAP FEATURE OVERLAP WITH FORMAL REFERENCE GROUP
   Subgroup                               Top 5 Features  Shared With Formal                      Unique To This Subgroup  Overlap With Formal %
     formal    punct_intensity, to, not, trump, char_ed                   20                                                               100.0
emoji-heavy happy, punct_intensity, love, char_ i , good                  13       slang_ratio, we, my, char_rea, amazing                   65.0
slang-heavy slang_ratio, punct_intensity, love, lol, bad                   8        slang_ratio, bet, char_ lol, bad, wtf                   40.0
    sarcasm not, at all, not at, idiots, punct_intensity                   6 at all, char_s! , fantastic, as if, for sure                   30.0
  saved table -> Table6b_SHAP_Subgroup_Comparison.csv

SHAP profiles saved for reuse in Notebook 15 (error taxonomy).


## Cell 7: Experiment 17 Part B — LIME on Confident Errors

Twenty of the most confident misclassifications, explained individually. Fixed
seed for reproducibility.

These are the most damaging errors in deployment: the model is wrong and certain
about it.

In [14]:
from lime.lime_text import LimeTextExplainer
import scipy.sparse as sp
import numpy as np

mis = pd.read_parquet(D / "misclassified_instances.parquet")
classes = sorted(tw_test["sentiment"].unique())

SOCIAL_COLS = ["emoji_density", "slang_ratio", "punct_intensity"]

# LIME feeds in perturbed text, so the engineered features have to be
# recomputed from that text — they cannot be looked up from the dataframe.
def build_features(texts):
    """Rebuild the exact 80,003-feature space the model was trained on."""
    texts = list(texts)

    if isinstance(transformers, tuple) and len(transformers) == 3:
        vec_w, vec_c, scaler = transformers
        Xw = vec_w.transform(texts)
        Xc = vec_c.transform(texts)
        tmp = pd.DataFrame({"text": texts})
        tmp = tag_subgroups(tmp, text_col="text")
        Xn = scaler.transform(tmp[SOCIAL_COLS].values)
        return sp.hstack([Xw, Xc, sp.csr_matrix(Xn)]).tocsr()

    if isinstance(transformers, tuple) and len(transformers) == 2:
        vec_w, scaler = transformers
        Xw = vec_w.transform(texts)
        tmp = pd.DataFrame({"text": texts})
        tmp = tag_subgroups(tmp, text_col="text")
        Xn = scaler.transform(tmp[SOCIAL_COLS].values)
        return sp.hstack([Xw, sp.csr_matrix(Xn)]).tocsr()

    return transformers.transform(texts)


def predict_proba_text(texts):
    return final_model.predict_proba(build_features(texts))


# sanity check before committing to 20 runs
test_out = predict_proba_text([str(mis.iloc[0]["text_clean"])])
print(f"Prediction function OK — output shape {test_out.shape}\n")

explainer = LimeTextExplainer(class_names=classes, random_state=SEED)
sample = mis.head(20)
lime_rows = []

print("Running LIME on 20 most confident errors...\n")
for i, (_, r) in enumerate(sample.iterrows(), 1):
    try:
        exp = explainer.explain_instance(
            str(r["text_clean"]), predict_proba_text,
            num_features=5, num_samples=500)
        feats = exp.as_list()
        lime_rows.append({
            "Subgroup":     r["subgroup"],
            "Actual":       r["y_true"],
            "Predicted":    r["y_pred"],
            "Confidence":   round(r["confidence"], 3),
            "Text":         str(r["text"])[:110],
            "Top Features": ", ".join(f"{f}({w:+.3f})" for f, w in feats[:3]),
        })
        if i <= 5:
            print(f"[{r['subgroup']}] {r['y_true']} -> {r['y_pred']} "
                  f"(conf {r['confidence']:.3f})")
            print(f"  {str(r['text'])[:95]}")
            print(f"  drivers: {', '.join(f'{f}({w:+.2f})' for f, w in feats[:3])}\n")
    except Exception as e:
        print(f"  instance {i} failed: {type(e).__name__}: {e}")

lime_df = pd.DataFrame(lime_rows)
save_result_table(lime_df, "Table6c_LIME_Confident_Errors")
print(f"Explained {len(lime_df)} of 20 instances.")

Prediction function OK — output shape (1, 3)

Running LIME on 20 most confident errors...

[other] neutral -> positive (conf 0.998)
  @user perfect @user
  drivers: perfect(-0.11)

[other] neutral -> positive (conf 0.997)
  Happy @user oppression day
  drivers: happy(-0.43), oppression(-0.10), day(-0.05)

[formal] neutral -> positive (conf 0.994)
  @user When will Melania do her "I have a dream" speech? I'm looking forward to it :)
  drivers: forward(-0.10), her(-0.05), looking(-0.05)

[formal] positive -> neutral (conf 0.994)
  The Reputation Doctor weighs in on Tony Romo #NFL @user joins @user on #TheMorningRush LISTEN:
  drivers: romo(+0.03), joins(+0.02), themorningrush(+0.01)

[other] neutral -> positive (conf 0.994)
  @user @user @user good mans
  drivers: good(-0.43), mans(+0.20)

  saved table -> Table6c_LIME_Confident_Errors.csv
Explained 20 of 20 instances.


## Cell 8: Table 6 — Ethical Risk Ranking

Combines everything into the five risk dimensions. Some inputs come from
notebooks that have not run yet — HCER from Notebook 13 and Fairness Drift from
Notebook 14 — so those dimensions are computed here with what is available and
recomputed at the end.

In [15]:
pred_final = load_predictions("exp10", "FinalModel")

wga, worst_sg = worst_group_accuracy(pred_final)
f1_var        = subgroup_f1_variance(pred_final)
conf          = confidence_risk(pred_final)
max_hcer      = conf[conf["Subgroup"] != "other"]["HCER"].max()

table4 = pd.read_csv(PATHS["results"] / "Table4_Subgroup_Fairness_Gap.csv")
mean_f1_gap = table4[table4["Subgroup"] != "formal"]["F1 Gap"].mean()

perf = score_performance_risk(wga, f1_var)
fair = score_fairness_risk(mean_f1_gap)
misc = score_misclassification_risk(max_hcer)
expl = 2                                   # provisional until Notebook 15
gen  = score_generalisation_risk(abs(gen_gap))

risk = ethical_risk_index(perf, fair, misc, expl, gen)

table6 = pd.DataFrame([{
    "Model":                     BEST_MODEL,
    "Main Dataset Macro F1":     res_tw["Macro F1"],
    "Cross-Dataset Macro F1":    res_se["Macro F1"],
    "Generalisation Gap":        gen_gap,
    "Highest Risk Subgroup":     worst_sg,
    "XAI Finding":               "See Table 6b for SHAP subgroup feature overlap",
    "WGA":                       wga,
    "Subgroup F1 Variance":      f1_var,
    "Mean F1 Gap":               mean_f1_gap,
    "Max HCER":                  max_hcer,
    **risk,
}])

print("="*70)
print("TABLE 6 — CROSS-DATASET ROBUSTNESS AND ETHICAL RISK RANKING")
print("="*70)
print(table6.T.to_string())

print(f"\nEthical Risk Score : {risk['Total Risk Score']} / 15")
print(f"Risk Level         : {risk['Final Ethical Risk Level']}")
print("\nExplainability Risk is provisional (scored 2) until Notebook 15")
print("computes cross-model SHAP agreement. Rerun this cell afterwards.")

save_result_table(table6, "Table6_Ethical_Risk_Ranking")

TABLE 6 — CROSS-DATASET ROBUSTNESS AND ETHICAL RISK RANKING
                                                                       0
Model                                                 LogisticRegression
Main Dataset Macro F1                                           0.545296
Cross-Dataset Macro F1                                          0.503831
Generalisation Gap                                              0.041465
Highest Risk Subgroup                                             formal
XAI Finding               See Table 6b for SHAP subgroup feature overlap
WGA                                                             0.592037
Subgroup F1 Variance                                            0.000241
Mean F1 Gap                                                    -0.089845
Max HCER                                                        0.496212
Performance Risk                                                       2
Fairness Risk                                                   

PosixPath('/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/results/Table6_Ethical_Risk_Ranking.csv')

## Cell 9: Done

In [16]:
print("="*62)
print("NOTEBOOK 9 COMPLETE — EXPERIMENTS 16 AND 17")
print("="*62)
print("\nCore experiments 1 to 17 are now finished.")
print("\nNovelty notebooks — each is independent and reads saved predictions:")
print("  NB10  Exp 18  Imbalance Correction        N1")
print("  NB11  Exp 19  Model Design                N2")
print("  NB12  Exp 20  Dual Framework Audit        N6")
print("  NB13  Exp 21  Confidence Risk             N4")
print("  NB14  Exp 22  Fairness Drift              N5")
print("  NB15  Exp 23  Error Taxonomy              N3")
print("  NB16  Exp 24  Fairness-Weighted Ensemble  N7")
print("\nNB13 is the shortest — roughly fifteen minutes for a full novelty point.")

NOTEBOOK 9 COMPLETE — EXPERIMENTS 16 AND 17

Core experiments 1 to 17 are now finished.

Novelty notebooks — each is independent and reads saved predictions:
  NB10  Exp 18  Imbalance Correction        N1
  NB11  Exp 19  Model Design                N2
  NB12  Exp 20  Dual Framework Audit        N6
  NB13  Exp 21  Confidence Risk             N4
  NB14  Exp 22  Fairness Drift              N5
  NB15  Exp 23  Error Taxonomy              N3
  NB16  Exp 24  Fairness-Weighted Ensemble  N7

NB13 is the shortest — roughly fifteen minutes for a full novelty point.
